In [1]:
import os
import numpy as np
import librosa
import pickle

In [2]:
preprocessed_dir = "data/preprocessed"
classes = ["copd", "healthy"]

features_dir = "data/features"
os.makedirs(features_dir, exist_ok=True)

In [3]:
def extract_features(y, sr=16000, n_mfcc=13):
    
    features = []
    
    #MFCCs (mean + std over frames)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    features.extend(np.mean(mfccs, axis=1))
    features.extend(np.std(mfccs, axis=1))
    
    #Spectral centroid
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)
    features.append(np.mean(spec_cent))
    features.append(np.std(spec_cent))
    
    #Spectral bandwidth
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    features.append(np.mean(spec_bw))
    features.append(np.std(spec_bw))
    
    #Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)
    features.append(np.mean(zcr))
    features.append(np.std(zcr))
    
    return np.array(features)

In [5]:
X = []
y_labels = []

for cls in classes:
    cls_dir = os.path.join(preprocessed_dir, cls)
    for fname in os.listdir(cls_dir):
        if not fname.endswith(".npy"):
            continue

        #Load preprocessed numeric array
        y_wave = np.load(os.path.join(cls_dir, fname))

        #Extract features
        feat = extract_features(y_wave, sr=16000)
        X.append(feat)
        y_labels.append(cls)

#Convert to arrays
X = np.array(X)
y_labels = np.array(y_labels)

print("Feature matrix shape:", X.shape)
print("Labels shape:", y_labels.shape)


Feature matrix shape: (1605, 32)
Labels shape: (1605,)


In [ ]:
#Save features for ML
np.save(os.path.join(features_dir, "X.npy"), X)
np.save(os.path.join(features_dir, "y.npy"), y_labels)

with open(os.path.join(features_dir, "dataset.pkl"), "wb") as f:
    pickle.dump({"X": X, "y": y_labels}, f)